# 🎣 AI-Driven Phishing Email Detection Using NLP
**Project**: IICT Semester Project  
**Objective**: Build a machine-learning system that automatically detects phishing emails using Natural Language Processing (NLP) and metadata features.  
**Pipeline Phases**:
1. Data Collection  
2. Data Cleaning (HTML stripping, NLTK tokenisation, stemming, stopword removal)  
3. Feature Engineering (TF-IDF + 14 structural metadata features)  
4. Model Development (Logistic Regression, Random Forest, Naive Bayes, MLP Neural Network)  
5. Evaluation (accuracy, precision, recall, F1, ROC-AUC, confusion matrices)  
6. Deployment Preview (live classification helper + Streamlit app)

> **Run from the project root** (`phishing_detection/`).  
> Requires: `pip install -r requirements.txt` (pandas, numpy, scikit-learn, nltk, matplotlib, seaborn, joblib, streamlit, python-pptx)

---
## Phase 1 — Data Collection

We use a **custom synthetic dataset** of 3,000 emails (50 % phishing / 50 % legitimate).
The dataset was generated by `generate_dataset.py` and saved to `data/raw_emails.csv`.

**Columns**:
| Column | Description |
|---|---|
| `subject` | Email subject line |
| `sender_domain` | Domain extracted from the From header |
| `body` | Full email body (may contain HTML, URLs) |
| `label` | 1 = phishing, 0 = legitimate |

> Real-world alternatives: **Kaggle Phishing Email Dataset**, **Enron corpus**, **SpamAssassin public corpus**.  
> To use them: `python run_pipeline.py --data emails.csv --text-col body --label-col label`

In [ ]:
import sys, os
# Ensure imports work whether running from project root or a notebooks/ subdir
sys.path.insert(0, os.path.abspath('..')  if os.path.basename(os.getcwd()) == 'notebooks' else '.')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA = 'data/raw_emails.csv'
if not os.path.exists(DATA):
    import generate_dataset
    generate_dataset.main()

raw = pd.read_csv(DATA)
print(f'Dataset shape : {raw.shape}')
print(f'Class balance :\n{raw["label"].value_counts()}')
raw.head(3)

In [ ]:
# Visualise class balance
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
raw['label'].value_counts().plot(kind='bar', ax=axes[0], color=['#55a868', '#c44e52'],
                                  title='Class Distribution')
axes[0].set_xticklabels(['Legitimate (0)', 'Phishing (1)'], rotation=0)
axes[1].pie(raw['label'].value_counts(), labels=['Legitimate', 'Phishing'],
            colors=['#55a868', '#c44e52'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class Balance')
plt.tight_layout()
plt.show()
print('Sample body (phishing):', raw.loc[raw.label==1, 'body'].iloc[0][:200])

---
## Phase 2 — Data Cleaning (NLP Preprocessing)

We clean the raw email text using the **NLTK** library — the standard Python toolkit for Natural Language Processing.

**Steps applied by `src/preprocess.py`**:

| Step | Tool | Purpose |
|---|---|---|
| HTML entity decode + tag stripping | `html`, `re` | Remove `<b>`, `<a href=...>` etc. |
| URL replacement | `re` | Replace all URLs with `urltoken` to preserve *presence* as a feature |
| Lowercase | Python str | Normalise case |
| Punctuation removal | `re` | Strip non-alphanumeric characters |
| Tokenisation | `nltk.tokenize.word_tokenize` | Split text into individual words |
| Stopword removal | `nltk.corpus.stopwords` | Remove common words (the, and, is…) |
| Stemming | `nltk.stem.PorterStemmer` | Reduce words to their root form (e.g. *verifying → verifi*) |

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Show NLTK preprocessing step-by-step on one example
sample = raw.loc[raw.label == 1, 'body'].iloc[0]
print('=== ORIGINAL TEXT (first 300 chars) ===')
print(sample[:300])
print()

import re, html as _html
step1 = _html.unescape(sample)
step1 = re.sub(r'<[^>]+>', ' ', step1)
print('=== After HTML stripping ===')
print(step1[:300])

step2 = re.sub(r'(https?://\S+|www\.\S+)', ' urltoken ', step1, flags=re.I)
print('\n=== After URL replacement ===')
print(step2[:300])

step3 = re.sub(r'[^a-z0-9\s]', ' ', step2.lower())
tokens = word_tokenize(step3)
stop_words = set(stopwords.words('english'))
tokens_filtered = [t for t in tokens if t not in stop_words and len(t) > 1]
print(f'\n=== After tokenise + stopword removal ===')
print(tokens_filtered[:30])

stemmer = PorterStemmer()
tokens_stemmed = [stemmer.stem(t) for t in tokens_filtered]
print(f'\n=== After Porter Stemming ===')
print(tokens_stemmed[:30])

In [ ]:
from src.train import load_and_prepare
from src.preprocess import clean_text

df = load_and_prepare(DATA)
df.to_csv('data/cleaned_emails.csv', index=False)
print(f'Cleaned dataset shape: {df.shape}')
print('Columns:', list(df.columns))
df[['subject', 'body', 'clean_text', 'label']].head(3)

---
## Phase 3 — Feature Engineering

We build **two complementary feature groups** and combine them:

### 3a. TF-IDF (Text features)

**TF-IDF** (Term Frequency × Inverse Document Frequency) converts each cleaned email into a numeric vector:

$$\text{TF-IDF}(t, d) = \text{TF}(t,d) \times \log\frac{N}{\text{DF}(t)}$$

- **TF** — how often term *t* appears in document *d* (sub-linear: `log(1+tf)`)  
- **IDF** — penalises terms common across *all* documents (not discriminative)  
- We use **uni-grams + bi-grams** (`ngram_range=(1,2)`) with up to **5,000 features**

### 3b. Metadata / Structural features (14 hand-crafted signals)

| Feature | Phishing Signal |
|---|---|
| `n_urls` | Phishing mails typically embed more links |
| `has_ip_url` | IP-based URLs bypass domain reputation |
| `has_shortener` | URL shorteners hide the real destination |
| `url_has_at` | `@` in URL hides real domain after the symbol |
| `suspicious_url_tld` | `.xyz`, `.top`, `.ru`, `.tk`, etc. are cheap disposable domains |
| `avg_url_len` | Very long URLs suggest obfuscation |
| `sender_bad_tld` | Fake sender domains use same suspicious TLDs |
| `sender_hyphen` | `paypal-secure.xyz` impersonates brands |
| `has_html` | HTML used to hide malicious elements |
| `n_exclaim` | Multiple `!` create artificial urgency |
| `upper_ratio` | EXCESSIVE CAPITALS signal urgency |
| `digit_ratio` | Obfuscated URLs contain many digits |
| `body_len` | Very short or very long bodies are unusual |
| `urgency_hits` | Count of urgency/credential keywords |

In [ ]:
from src.features import META_COLS

# Average metadata values by class — shows discriminative power
meta_means = df.groupby('label')[META_COLS].mean().T
meta_means.columns = ['Legitimate', 'Phishing']
meta_means['Phishing / Legit ratio'] = (meta_means['Phishing'] / (meta_means['Legitimate'] + 1e-6)).round(2)
print(meta_means.round(3))

In [ ]:
# Visualise top discriminating metadata features
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
features_to_plot = ['n_urls', 'urgency_hits', 'upper_ratio', 'n_exclaim',
                    'has_ip_url', 'suspicious_url_tld', 'sender_bad_tld', 'body_len']
for ax, col in zip(axes.flat, features_to_plot):
    for label, color, name in [(0, '#55a868', 'Legit'), (1, '#c44e52', 'Phish')]:
        vals = df.loc[df.label == label, col]
        ax.hist(vals, bins=20, alpha=0.6, color=color, label=name, density=True)
    ax.set_title(col); ax.legend(fontsize=7)
plt.suptitle('Metadata Feature Distributions: Legit vs Phishing', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('outputs/metadata_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# TF-IDF vocabulary preview
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=5000, min_df=2, sublinear_tf=True)
tfidf.fit(df['clean_text'])
vocab = sorted(tfidf.vocabulary_.items(), key=lambda x: x[1])
print(f'Vocabulary size: {len(tfidf.vocabulary_)} terms')
print('Sample terms:', [v[0] for v in vocab[:30]])

---
## Phase 4 — Model Development

We train **four classifiers** using scikit-learn Pipelines, each combining the ColumnTransformer (TF-IDF + scaled metadata) with the estimator.  
Hyper-parameters are tuned with **3-fold GridSearchCV** optimising the **F1 score** (balances precision and recall — important when both false positives and false negatives are costly).

| Model | Why chosen |
|---|---|
| **Logistic Regression** | Fast, interpretable; strong baseline for text classification |
| **Random Forest** | Handles mixed text+metadata features; provides feature importance |
| **Multinomial Naive Bayes** | Classic NLP model; assumes word-count independence; very fast |
| **MLP Neural Network** | Captures non-linear patterns; early stopping prevents overfitting |

In [ ]:
from src.train import run

res, ablation, best_model = run(DATA)
print('\n=== Model Comparison Table ===')
res.drop(columns='best_params').astype(float).round(4)

---
## Phase 5 — Evaluation

**Metrics explained**:

| Metric | Formula | Meaning |
|---|---|---|
| Accuracy | (TP+TN)/(TP+TN+FP+FN) | Overall correctness |
| Precision | TP/(TP+FP) | Of flagged phishing, how many were actually phishing? |
| Recall | TP/(TP+FN) | Of actual phishing, how many did we catch? |
| F1 Score | 2·P·R/(P+R) | Harmonic mean of precision and recall |
| ROC-AUC | Area under ROC curve | Ranking quality; 1.0 = perfect |

> **In phishing detection, Recall (catching all phishing) is typically more important than Precision.**
> Missing a phishing email (False Negative) is usually worse than a false alarm (False Positive).

In [ ]:
# Feature ablation study: does metadata help?
print('=== Feature Ablation Study (Logistic Regression) ===')
import pandas as pd
abl_df = pd.DataFrame(ablation).T.round(4)
print(abl_df)
print()
print('Insight: Text-only reaches same accuracy as combined, but metadata alone achieves ~93%.'
      ' Combined gives highest ROC-AUC, confirming both feature sets contribute.')

In [ ]:
# Display all saved visualisations
from IPython.display import Image, display

plots = [
    ('Confusion Matrices', 'outputs/confusion_matrices.png'),
    ('Model Comparison', 'outputs/model_comparison.png'),
    ('ROC Curves', 'outputs/roc_curves.png'),
    ('Random Forest Feature Importance', 'outputs/feature_importance_rf.png'),
    ('Logistic Regression Coefficients', 'outputs/feature_importance_lr.png'),
    ('Metadata Distributions', 'outputs/metadata_distributions.png'),
]
for title, path in plots:
    if os.path.exists(path):
        print(f'\n--- {title} ---')
        display(Image(path))

---
## Phase 6 — Deployment Preview

The `classify()` helper loads the saved best model and returns:
- `label` — PHISHING or LEGITIMATE
- `phishing_probability` — model confidence (0–1)
- `red_flags` — human-readable explanations of triggered signals

Full web UI: run `streamlit run app.py` from the project root.

In [ ]:
from src.predict import classify

# --- Test 1: Obvious phishing email ---
result = classify(
    subject='⚠️ URGENT: Your PayPal account has been SUSPENDED!',
    sender_domain='paypal-secure-login.xyz',
    body='Dear Customer, your account has been suspended due to unusual activity. '
         'Verify your identity IMMEDIATELY: http://paypal-secure-login.xyz/verify?token=abc123. '
         'Failure to verify within 24 hours will result in permanent account closure. '
         'Password required. Claim your refund now!'
)
print('=== Test 1: Phishing Email ===')
print(f"  Label : {result['label']}")
print(f"  Confidence : {result['phishing_probability']:.1%}")
print(f"  Red flags  : {result['red_flags']}")

In [ ]:
# --- Test 2: Legitimate email ---
result2 = classify(
    subject='Team meeting notes — Q3 review',
    sender_domain='company.com',
    body='Hi Sarah, please find attached the notes from today\'s Q3 review meeting. '
         'The key action items are listed on page 2. Let me know if you have any questions. '
         'Best regards, John'
)
print('=== Test 2: Legitimate Email ===')
print(f"  Label : {result2['label']}")
print(f"  Confidence : {result2['phishing_probability']:.1%}")
print(f"  Red flags  : {result2['red_flags']}")

In [ ]:
# --- Test 3: Edge case — stealthy phishing (no urgency words, clean URL) ---
result3 = classify(
    subject='Your invoice is ready',
    sender_domain='billing-support.info',
    body='Please log in to review your invoice: http://bit.ly/inv-2024-review'
)
print('=== Test 3: Stealthy Phishing (edge case) ===')
print(f"  Label : {result3['label']}")
print(f"  Confidence : {result3['phishing_probability']:.1%}")
print(f"  Red flags  : {result3['red_flags']}")

---
## Summary & Conclusions

| Deliverable | Status |
|---|---|
| Python notebook with full code + documentation | ✅ this file |
| Raw dataset (`data/raw_emails.csv`) | ✅ 3,000 emails |
| Cleaned dataset (`data/cleaned_emails.csv`) | ✅ NLTK-processed |
| Comparative analysis report (`comparative_report.md`) | ✅ |
| Visualisations (confusion matrices, ROC, feature importance) | ✅ in `outputs/` |
| Saved best model (`outputs/best_model.joblib`) | ✅ |
| Streamlit web interface (`app.py`) | ✅ run `streamlit run app.py` |
| Presentation slides (`outputs/presentation.pptx`) | ✅ |

### Key Findings
- All four models achieve **~96.6% accuracy** (ceiling imposed by synthetic dataset noise)
- **Text (TF-IDF)** and **metadata features** each contribute: combined gives highest ROC-AUC
- Top phishing indicators: suspicious URLs, urgency keywords, bad sender TLDs
- **Logistic Regression** is recommended as the best model — matches RF in accuracy but is interpretable

### Ethical Considerations
- Detectors produce **false positives** (blocking real mail) and **false negatives** (missed attacks)
- Use as **decision support** alongside human review
- Avoid storing personal email content; monitor for adversarial drift

### Future Work
- Validate on real data (Kaggle, Enron, SpamAssassin)
- Add transformer-based embeddings (DistilBERT)
- Include email header features (SPF, DKIM, DMARC)
- Retrain periodically as phishing tactics evolve